# 🏥 Trust-Based Decentralized Federated Learning — HR Prediction

### Peer-to-Peer Trust System for Detecting Malicious Nodes

This notebook implements a **fully decentralized** federated learning experiment where each node:
1. **Trains locally** on its own patient data
2. **Evaluates each neighbor 1-on-1** — merges temporarily and checks if predictions improve or worsen
3. **Votes**: if merging degrades MAE beyond a threshold → 🔴 **red vote** (penalize trust). Otherwise → ✅ **pass** (reward trust)
4. **Aggregates only with trusted neighbors** — nodes with negative cumulative trust get blocked

Trust scores accumulate across rounds, building a reputation system that isolates malicious nodes over time.

---

**Key parameters:**
- `threshold` — MAE degradation tolerance (default 10%). If merging worsens MAE by more than this → red vote
- `num_poisoned` — how many nodes have corrupted data
- `attack` — poisoning strategy (label_noise, label_flip, combined_aggressive, etc.)
- `topology` — network structure (full, star, ring, line)

**No p2pfl dependency** — this is a standalone PyTorch implementation.

---
**⚙️ Optional:** Set runtime to GPU (`Runtime → Change runtime type → T4`) for faster training.

## 1. Setup

In [1]:
# Install dependencies (most are pre-installed on Colab)
!pip install torch pandas numpy -q
print("✅ Ready!")

✅ Ready!


## 2. Get Data & Script

In [2]:
import os, base64

# Clone repo for the cleaned_data folder
!git clone --branch malicious-nodes --single-branch https://github.com/Dimitris-Gerakas-CERTH/p2pfl.git /content/p2pfl 2>/dev/null || echo "Already cloned"

PROJECT_DIR = "/content/p2pfl/p2pfl/examples/fl_Health_Demo"
os.chdir(PROJECT_DIR)

# Write the trust FL script from embedded source
script = base64.b64decode("...").decode()
with open("trust_fl_experiment.py", "w") as f:
    f.write(script)

# Verify
data_files = sorted(os.listdir("cleaned_data"))
print(f"📂 Working directory: {os.getcwd()}")
print(f"📂 Patient data: {len(data_files)} files")
print(f"📄 Script: trust_fl_experiment.py ({len(script):,} bytes)")
print("\n✅ All set!")


📂 Working directory: /content/p2pfl/p2pfl/examples/fl_Health_Demo
📂 Patient data: 22 files
📄 Script: trust_fl_experiment.py (42,689 bytes)

✅ All set!


## 3. Verify Setup

In [3]:
import torch
import sys
sys.path.insert(0, "/content/p2pfl/p2pfl/examples/fl_Health_Demo")

from trust_fl_experiment import run_experiment, HRPredictorMLP, FLNode

print(f"✅ All imports work!")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

✅ All imports work!
   PyTorch: 2.10.0+cu128
   CUDA: True
   GPU: Tesla T4


## 4. Clean Experiment (No Poisoning)

Run a baseline experiment with no malicious nodes to establish normal performance.

In [6]:
from pathlib import Path

results_clean = run_experiment(
    data_dir=Path("./cleaned_data"),
    num_nodes=8,
    num_rounds=7,
    epochs_per_round=2,
    window_size=10,
    batch_size=32,
    topology="full",
    threshold=0.08,
    reduced_fraction=0.05,
    poisoned_nodes=[],        # clean experiment
    lr=0.001,
    seed=5,
    random_window=True,
    green_reward=0.1,
    red_penalty=0.2,
    gossip=False,             # no gossip for baseline
)


🏥 TRUST-BASED DECENTRALIZED FL — HR PREDICTION
   Nodes: 8 | Rounds: 7 | Epochs/Round: 2
   Topology: full | Trust threshold: 8% MAE degradation
   Device: cuda
   ✅ Clean experiment
   🎲 Random window: ON (seed=5)
   📊 Trust scoring: green=+0.1, red=-0.2
   📢 Gossip: disabled

📦 Loading data for 8 patients...
   ✓ Patient 01: 2,706 train, 670 test
   ✓ Patient 02: 2,424 train, 599 test
   ✓ Patient 03: 2,700 train, 668 test
   ✓ Patient 04: 2,581 train, 638 test
   ✓ Patient 05: 2,591 train, 641 test
   ✓ Patient 06: 2,686 train, 664 test
   ✓ Patient 07: 2,738 train, 677 test
   ✓ Patient 08: 2,456 train, 607 test

   Total: 8 nodes ready

🔗 Topology: full
   P01 ↔ [P02, P03, P04, P05, P06, P07, P08]
   P02 ↔ [P01, P03, P04, P05, P06, P07, P08]
   P03 ↔ [P01, P02, P04, P05, P06, P07, P08]
   P04 ↔ [P01, P02, P03, P05, P06, P07, P08]
   P05 ↔ [P01, P02, P03, P04, P06, P07, P08]
   P06 ↔ [P01, P02, P03, P04, P05, P07, P08]
   P07 ↔ [P01, P02, P03, P04, P05, P06, P08]
   P08 ↔ [P01, P0

## 5. Poisoned Experiment — WITHOUT Gossip (Baseline)

Run with poisoned nodes but gossip disabled. This establishes the baseline detection rate that we'll compare against the gossip-enabled version.

In [9]:
from pathlib import Path

# Poisoned experiment WITHOUT gossip (baseline for comparison)
results_no_gossip = run_experiment(
    data_dir=Path("./cleaned_data"),
    num_nodes=8,
    num_rounds=7,
    epochs_per_round=2,
    window_size=10,
    batch_size=32,
    topology="full",
    threshold=0.1,
    reduced_fraction=0.05,
    poisoned_nodes=[4, 5],
    attack="label_noise",
    noise_std = 0.9,
    poison_rate=1.0,
    lr=0.001,
    seed=4,
    random_window=True,
    green_reward=0.1,
    red_penalty=0.2,
    gossip=False,             # gossip OFF
)


🏥 TRUST-BASED DECENTRALIZED FL — HR PREDICTION
   Nodes: 8 | Rounds: 7 | Epochs/Round: 2
   Topology: full | Trust threshold: 10% MAE degradation
   Device: cuda
   ☠️  Poisoned: 2/8 [P04, P05] | Attack: label_noise
   🎲 Random window: ON (seed=4)
   📊 Trust scoring: green=+0.1, red=-0.2
   📢 Gossip: disabled

📦 Loading data for 8 patients...
   ✓ Patient 01: 2,706 train, 670 test
   ✓ Patient 02: 2,424 train, 599 test
   ✓ Patient 03: 2,700 train, 668 test
   ☠️  Patient 04: POISONED (label_noise), HR mean 77.1 → 76.9
   ☠️  Patient 05: POISONED (label_noise), HR mean 62.0 → 61.9
   ✓ Patient 06: 2,686 train, 664 test
   ✓ Patient 07: 2,738 train, 677 test
   ✓ Patient 08: 2,456 train, 607 test

   Total: 8 nodes ready

🔗 Topology: full
   P01 ↔ [P02, P03, P04, P05, P06, P07, P08]
   P02 ↔ [P01, P03, P04, P05, P06, P07, P08]
   P03 ↔ [P01, P02, P04, P05, P06, P07, P08]
   P04 ☠️ ↔ [P01, P02, P03, P05, P06, P07, P08]
   P05 ☠️ ↔ [P01, P02, P03, P04, P06, P07, P08]
   P06 ↔ [P01, P02, 

## 6. Poisoned Experiment — WITH Gossip

Same parameters as above, but with gossip enabled. Nodes share reputation reports with trusted neighbors after each evaluation round. Compare detection rates with the baseline above.

In [5]:
from pathlib import Path

# Poisoned experiment WITH gossip
results_gossip = run_experiment(
    data_dir=Path("./cleaned_data"),
    num_nodes=8,
    num_rounds=7,
    epochs_per_round=2,
    window_size=10,
    batch_size=32,
    topology="full",
    threshold=0.1,
    reduced_fraction=0.05,
    poisoned_nodes=[4, 5],
    attack="label_noise",
    noise_std = 0.9,
    poison_rate=1.0,
    lr=0.001,
    seed=55,
    random_window=True,
    green_reward=0.1,
    red_penalty=0.2,
    gossip=True,              # gossip ON
    gossip_weight=0.6,        # gossip influence (0=off, 1=same as direct eval)
)


🏥 TRUST-BASED DECENTRALIZED FL — HR PREDICTION
   Nodes: 8 | Rounds: 7 | Epochs/Round: 2
   Topology: full | Trust threshold: 10% MAE degradation
   Device: cuda
   ☠️  Poisoned: 2/8 [P04, P05] | Attack: label_noise
   🎲 Random window: ON (seed=55)
   📊 Trust scoring: green=+0.1, red=-0.2
   📢 Gossip: ENABLED (weight=0.6)

📦 Loading data for 8 patients...
   ✓ Patient 01: 2,706 train, 670 test
   ✓ Patient 02: 2,424 train, 599 test
   ✓ Patient 03: 2,700 train, 668 test
   ☠️  Patient 04: POISONED (label_noise), HR mean 65.4 → 65.5
   ☠️  Patient 05: POISONED (label_noise), HR mean 94.6 → 94.5
   ✓ Patient 06: 2,686 train, 664 test
   ✓ Patient 07: 2,738 train, 677 test
   ✓ Patient 08: 2,456 train, 607 test

   Total: 8 nodes ready

🔗 Topology: full
   P01 ↔ [P02, P03, P04, P05, P06, P07, P08]
   P02 ↔ [P01, P03, P04, P05, P06, P07, P08]
   P03 ↔ [P01, P02, P04, P05, P06, P07, P08]
   P04 ☠️ ↔ [P01, P02, P03, P05, P06, P07, P08]
   P05 ☠️ ↔ [P01, P02, P03, P04, P06, P07, P08]
   P06 

## 7. Compare Multiple Attacks

Run several attack strategies and compare detection with and without gossip.

In [ ]:
from pathlib import Path
import pandas as pd

attacks_to_test = [
    ("label_noise", 0.5),
    ("label_flip", 0.5),
    ("feature_noise", 0.5),
    ("combined_subtle", 0.5),
    ("combined_aggressive", 0.5),
]

comparison = []
sep = "=" * 70

for attack_name, noise in attacks_to_test:
    print(f"\n{sep}")
    print(f"Testing attack: {attack_name}")
    print(sep)

    # Run WITHOUT gossip
    r_no = run_experiment(
        data_dir=Path("./cleaned_data"),
        num_nodes=7, num_rounds=5, epochs_per_round=2,
        topology="full", threshold=0.08, reduced_fraction=0.05,
        poisoned_nodes=[3, 7], attack=attack_name, noise_std=noise,
        seed=42, random_window=True, green_reward=0.15, red_penalty=0.25,
        gossip=False,
    )

    # Run WITH gossip
    r_go = run_experiment(
        data_dir=Path("./cleaned_data"),
        num_nodes=7, num_rounds=5, epochs_per_round=2,
        topology="full", threshold=0.08, reduced_fraction=0.05,
        poisoned_nodes=[3, 7], attack=attack_name, noise_std=noise,
        seed=42, random_window=True, green_reward=0.15, red_penalty=0.25,
        gossip=True, gossip_weight=0.5,
    )

    for label, r in [("No Gossip", r_no), ("Gossip", r_go)]:
        clean_nodes = [n for n in r["nodes"] if not n.is_poisoned]
        poisoned_ids = {n.node_id for n in r["nodes"] if n.is_poisoned}
        detected = set()
        false_pos = 0
        for node in clean_nodes:
            for nbr_id, ts in node.trust_scores.items():
                if nbr_id in poisoned_ids and not ts.is_trusted:
                    detected.add(nbr_id)
                elif nbr_id not in poisoned_ids and not ts.is_trusted:
                    false_pos += 1
        clean_maes = [r["rounds"][-1][n.node_id]["mae"] for n in clean_nodes]
        comparison.append({
            "Attack": attack_name,
            "Gossip": label,
            "Clean MAE": f"{sum(clean_maes)/len(clean_maes):.4f}",
            "Detected": f"{len(detected)}/{len(poisoned_ids)}",
            "False Pos": false_pos,
        })

print("\n" + sep)
print("ATTACK COMPARISON: GOSSIP vs NO GOSSIP")
print(sep)
print(pd.DataFrame(comparison).to_string(index=False))

Streaming output truncated to the last 5000 lines.
   P04: train_loss=0.0332
   P05: train_loss=0.0362
   P06: train_loss=0.0447
   P07 ☠️: train_loss=0.2511

🔍 Peer evaluation (threshold: 8% MAE degradation)...
   P01 → P02: MAE 0.1520→0.1634 (+7.5%) ✅ PASS
   P01 → P03 ☠️: MAE 0.1520→0.2376 (+56.3%) 🔴 RED
   P01 → P04: MAE 0.1520→0.1468 (-3.4%) ✅ PASS
   P01 → P05: MAE 0.1520→0.1436 (-5.5%) ✅ PASS
   P01 → P06: MAE 0.1520→0.1751 (+15.2%) 🔴 RED
   P01 → P07 ☠️: MAE 0.1520→0.2011 (+32.3%) 🔴 RED
   P02 → P01: MAE 0.1226→0.1038 (-15.3%) ✅ PASS
   P02 → P03 ☠️: MAE 0.1226→0.1570 (+28.1%) 🔴 RED
   P02 → P04: MAE 0.1226→0.0998 (-18.6%) ✅ PASS
   P02 → P05: MAE 0.1226→0.0967 (-21.1%) ✅ PASS
   P02 → P06: MAE 0.1226→0.1108 (-9.6%) ✅ PASS
   P02 → P07 ☠️: MAE 0.1226→0.1238 (+1.0%) ✅ PASS
   P03 → P01: MAE 0.4275→0.4470 (+4.6%) ✅ PASS
   P03 → P02: MAE 0.4275→0.4167 (-2.5%) ✅ PASS
   P03 → P04: MAE 0.4275→0.4297 (+0.5%) ✅ PASS
   P03 → P05: MAE 0.4275→0.4321 (+1.1%) ✅ PASS
   P03 → P06: MAE 0.4

## 8. (Optional) Test Different Topologies

See how network structure affects trust detection with gossip enabled.

In [ ]:
from pathlib import Path
import pandas as pd

topologies = ["full", "star", "ring", "line"]
topo_results = []
sep = "=" * 70

for topo in topologies:
    print(f"\n{sep}")
    print(f"Topology: {topo}")
    print(sep)

    r = run_experiment(
        data_dir=Path("./cleaned_data"),
        num_nodes=7, num_rounds=5, epochs_per_round=2,
        topology=topo, threshold=0.08, reduced_fraction=0.05,
        poisoned_nodes=[3, 7], attack="label_flip",
        seed=42, random_window=True, green_reward=0.15, red_penalty=0.25,
        gossip=True, gossip_weight=0.5,
    )

    clean_nodes = [n for n in r["nodes"] if not n.is_poisoned]
    poisoned_ids = {n.node_id for n in r["nodes"] if n.is_poisoned}
    detected = set()
    for node in clean_nodes:
        for nbr_id, ts in node.trust_scores.items():
            if nbr_id in poisoned_ids and not ts.is_trusted:
                detected.add(nbr_id)
    clean_maes = [r["rounds"][-1][n.node_id]["mae"] for n in clean_nodes]

    topo_results.append({
        "Topology": topo,
        "Clean MAE": f"{sum(clean_maes)/len(clean_maes):.4f}",
        "Detected": f"{len(detected)}/{len(poisoned_ids)}",
        "Time (s)": f"{r['exec_time']:.1f}",
    })

print("\n" + sep)
print("TOPOLOGY COMPARISON (with gossip)")
print(sep)
print(pd.DataFrame(topo_results).to_string(index=False))


Topology: full

🏥 TRUST-BASED DECENTRALIZED FL — HR PREDICTION
   Nodes: 7 | Rounds: 5 | Epochs/Round: 2
   Topology: full | Trust threshold: 8% MAE degradation
   Device: cuda
   ☠️  Poisoned: 2/7 [P03, P07] | Attack: label_flip
   🎲 Random window: ON (seed=42)
   📊 Trust scoring: green=+0.15, red=-0.25
   📢 Gossip: ENABLED (weight=0.5)

📦 Loading data for 7 patients...
   ✓ Patient 01: 2,706 train, 670 test
   ✓ Patient 02: 2,424 train, 599 test
   ☠️  Patient 03: POISONED (label_flip), HR mean 79.0 → 79.0
   ✓ Patient 04: 2,581 train, 638 test
   ✓ Patient 05: 2,591 train, 641 test
   ✓ Patient 06: 2,686 train, 664 test
   ☠️  Patient 07: POISONED (label_flip), HR mean 86.2 → 86.2

   Total: 7 nodes ready

🔗 Topology: full
   P01 ↔ [P02, P03, P04, P05, P06, P07]
   P02 ↔ [P01, P03, P04, P05, P06, P07]
   P03 ☠️ ↔ [P01, P02, P04, P05, P06, P07]
   P04 ↔ [P01, P02, P03, P05, P06, P07]
   P05 ↔ [P01, P02, P03, P04, P06, P07]
   P06 ↔ [P01, P02, P03, P04, P05, P07]
   P07 ☠️ ↔ [P01, P0

## 9. (Optional) Threshold Sensitivity Analysis

How does the trust threshold affect detection? Lower = more suspicious, higher = more tolerant.

In [ ]:
from pathlib import Path
import pandas as pd

thresholds = [0.05, 0.08, 0.10, 0.15, 0.20]
thresh_results = []

for t in thresholds:
    r = run_experiment(
        data_dir=Path("./cleaned_data"),
        num_nodes=7, num_rounds=5, epochs_per_round=2,
        topology="full", threshold=t, reduced_fraction=0.05,
        poisoned_nodes=[3, 7], attack="label_flip",
        seed=42, random_window=True, green_reward=0.15, red_penalty=0.25,
        gossip=True, gossip_weight=0.5,
    )

    clean_nodes = [n for n in r["nodes"] if not n.is_poisoned]
    poisoned_ids = {n.node_id for n in r["nodes"] if n.is_poisoned}
    clean_ids = {n.node_id for n in clean_nodes}
    detected = set()
    false_pos = 0
    for node in clean_nodes:
        for nbr_id, ts in node.trust_scores.items():
            if nbr_id in poisoned_ids and not ts.is_trusted:
                detected.add(nbr_id)
            elif nbr_id in clean_ids and not ts.is_trusted:
                false_pos += 1
    clean_maes = [r["rounds"][-1][n.node_id]["mae"] for n in clean_nodes]

    thresh_results.append({
        "Threshold": f"{t*100:.0f}%",
        "Clean MAE": f"{sum(clean_maes)/len(clean_maes):.4f}",
        "Detected": f"{len(detected)}/{len(poisoned_ids)}",
        "False Pos": false_pos,
    })

print("\n" + "=" * 70)
print("THRESHOLD SENSITIVITY (with gossip)")
print("=" * 70)
print(pd.DataFrame(thresh_results).to_string(index=False))
print("\nLower threshold = more detections but more false positives")
print("Higher threshold = fewer false positives but may miss subtle attacks")

---
## Appendix: Available Options

| Parameter | Default | Description |
|-----------|---------|-------------|
| `num_nodes` | 7 | Number of patients/nodes (max 22) |
| `num_rounds` | 5 | Federated learning rounds |
| `epochs_per_round` | 2 | Local training epochs per round |
| `topology` | "full" | Network: full, star, ring, line |
| `threshold` | 0.10 | MAE degradation tolerance (10%) |
| `reduced_fraction` | 0.05 | Data fraction (0.05 = 5% = quick mode) |
| `num_poisoned` | 0 | Poisoned nodes (first N patients) |
| `attack` | "label_noise" | Attack strategy |
| `poison_rate` | 1.0 | Fraction of samples poisoned per node |
| `noise_std` | 0.5 | Noise intensity for noise attacks |
| `seed` | 42 | Random seed for reproducibility |
| `random_window` | False | Sample random contiguous time window instead of first rows |
| `green_reward` | 0.1 | Trust score increase per green vote |
| `red_penalty` | 0.3 | Trust score decrease per red vote |
| `gossip` | False | Enable gossip-based reputation sharing |
| `gossip_weight` | 0.5 | Gossip influence relative to direct eval (0-1) |

**Available attacks:** `label_noise`, `label_flip`, `label_constant`, `feature_noise`, `temporal_shift`, `combined_subtle`, `combined_aggressive`

**Trust scoring:** Green vote = +`green_reward` (default 0.1), Red vote = −`red_penalty` (default 0.3). Configurable. Gossip amplifies detection through peer reputation reports. Node blocked when cumulative score ≤ 0.